# shopextract — E-Commerce Product Intelligence

**Extract, analyze, validate, and monitor product data from any online store.**

`shopextract` is a Python library that turns any e-commerce URL into structured product data. It automatically detects the underlying platform, discovers product pages, and extracts inventory using a tiered strategy — preferring fast API calls, falling back to intelligent crawling when needed.

### What you can do with it

| Capability | Description |
|---|---|
| **Platform Detection** | Identify Shopify, WooCommerce, Magento, BigCommerce, Shopware, or custom stores |
| **Product Extraction** | Pull structured product data (title, price, images, GTIN, variants) from any store |
| **Catalog Analysis** | Price distributions, brand breakdowns, completeness scoring |
| **Product Matching** | Fuzzy title matching and exact GTIN lookup across catalogs |
| **Marketplace Validation** | Check readiness for Google Shopping, idealo, Amazon, eBay |
| **Price Monitoring** | Track price changes, new arrivals, and discontinued products over time |
| **Multi-Format Export** | CSV, JSON, Google Shopping XML, Pandas DataFrame, Parquet |

This notebook walks through every major feature with live examples.

In [ ]:
import sys
sys.path.insert(0, "../src")

import shopextract

print(f"shopextract v{shopextract.__version__}")
print(f"Platforms supported: {', '.join(p.value for p in shopextract.Platform)}")
print(f"Extraction tiers:   {', '.join(t.value for t in shopextract.ExtractionTier)}")

---

## 1. Platform Detection

`detect()` probes a URL using HTTP headers, API endpoints, and HTML content analysis to identify the e-commerce platform. It returns a `PlatformResult` with the detected platform, a confidence score (0.0 - 1.0), and the specific signals that triggered detection.

Detection runs in under 3 seconds for most stores — no browser needed.

In [ ]:
# Detect a Shopify demo store
result = await shopextract.detect("https://hydrogen-preview.myshopify.com")
print(f"Platform:   {result.platform.value}")
print(f"Confidence: {result.confidence:.0%}")
print(f"Signals:    {', '.join(result.signals)}")

In [ ]:
# Detect a Magento test store
result_2 = await shopextract.detect("https://magento.softwaretestingboard.com")
print(f"Platform:   {result_2.platform.value}")
print(f"Confidence: {result_2.confidence:.0%}")
print(f"Signals:    {', '.join(result_2.signals)}")

---

## 2. Product Extraction

`extract()` is the main workhorse. Given a store URL, it:

1. **Detects** the platform (or accepts a pre-detected one)
2. **Discovers** product URLs via APIs, sitemaps, or crawling
3. **Extracts** product data using a tiered fallback chain:
   - **Tier 1 — API**: Platform-native JSON APIs (fastest, most reliable)
   - **Tier 2 — UnifiedCrawl**: JSON-LD, OpenGraph, markdown parsing from a single page fetch
   - **Tier 3 — CSS**: Browser-based extraction with CSS selectors
4. **Normalizes** everything into a unified `Product` dataclass

Each product includes: title, price, currency, description, images, SKU, GTIN, vendor, variants, and more.

In [ ]:
# Extract products from Shopify demo store (limited to 5 for demo)
result = await shopextract.extract("https://hydrogen-preview.myshopify.com", max_urls=5)
print(f"Platform:       {result.platform.value}")
print(f"Extraction Tier: {result.tier.value}")
print(f"Quality Score:  {result.quality_score:.2f}")
print(f"Products Found: {result.product_count}")
print(f"URLs Attempted: {result.urls_attempted}")
print(f"URLs Succeeded: {result.urls_succeeded}")

In [ ]:
# Display the first 5 products as a formatted table
import pandas as pd
from dataclasses import asdict

products = result.products
products_as_dicts = [asdict(p) for p in products]

df_display = pd.DataFrame([
    {
        "Title": p.title[:50] + ("..." if len(p.title) > 50 else ""),
        "Price": f"{p.price} {p.currency}",
        "Vendor": p.vendor or "—",
        "SKU": p.sku or "—",
        "In Stock": p.in_stock,
        "Image": (p.image_url[:60] + "...") if p.image_url else "—",
    }
    for p in products[:5]
])

df_display

---

## 3. Catalog Analysis

The analysis module computes aggregate statistics from product data — no network calls, pure computation. Feed it a list of product dicts and get back price ranges, brand distributions, completeness scores, and more.

In [ ]:
# Compute catalog statistics
stats = shopextract.analyze_products(products_as_dicts)

print(f"Total Products:     {stats.total_products}")
print(f"Price Range:        ${stats.price_range[0]:.2f} — ${stats.price_range[1]:.2f}")
print(f"Average Price:      ${stats.avg_price:.2f}")
print(f"Median Price:       ${stats.median_price:.2f}")
print(f"In Stock:           {stats.in_stock}")
print(f"Out of Stock:       {stats.out_of_stock}")
print(f"Has GTIN:           {stats.has_gtin}")
print(f"Has Images:         {stats.has_images}")
print(f"Completeness Score: {stats.completeness_score:.1%}")

if stats.currencies:
    print(f"\nCurrencies: {dict(stats.currencies)}")

In [ ]:
# Price distribution across buckets
dist = shopextract.price_distribution(products_as_dicts)

print("Price Distribution")
print("=" * 35)
for bucket, count in dist.items():
    bar = "█" * count
    print(f"  ${bucket:>10s}  {count:>3d}  {bar}")

In [ ]:
# Brand breakdown — percentage of catalog per brand/vendor
brands = shopextract.brand_breakdown(products_as_dicts)

if brands:
    print("Top 10 Brands by Catalog Share")
    print("=" * 40)
    for brand, pct in list(brands.items())[:10]:
        bar = "█" * int(pct / 2)
        print(f"  {brand:<25s} {pct:5.1f}%  {bar}")
else:
    print("No brand/vendor data available in this catalog.")

---

## 4. Product Matching and Comparison

When you have product data from multiple sources, `fuzzy_match()` finds the same product across catalogs using title similarity. `match_gtin()` does exact lookups by GTIN, EAN, UPC, or SKU.

This is useful for competitive price analysis, catalog deduplication, and supplier matching.

In [ ]:
# Simulate two store catalogs with overlapping products
store_a = [
    {"title": "Classic Percale Sheet Set - White",  "price": "149.00", "currency": "USD", "gtin": "0850031234567"},
    {"title": "Luxe Sateen Duvet Cover - Cream",    "price": "199.00", "currency": "USD", "gtin": "0850031234890"},
    {"title": "Down Alternative Comforter",          "price": "249.00", "currency": "USD", "gtin": "0850031234111"},
    {"title": "Waffle Bath Towel Set",               "price": "79.00",  "currency": "USD"},
]

store_b = [
    {"title": "Classic Percale Sheet Set (White)",   "price": "139.00", "currency": "USD", "gtin": "0850031234567"},
    {"title": "Luxe Sateen Duvet Cover, Cream",      "price": "209.00", "currency": "USD"},
    {"title": "Organic Cotton Pillowcase Pair",      "price": "49.00",  "currency": "USD"},
    {"title": "Waffle Towel Bath Set",               "price": "69.00",  "currency": "USD"},
]

# Fuzzy match by title
matches = shopextract.fuzzy_match(store_a, store_b, threshold=0.7)

print(f"Found {len(matches)} matching products across stores\n")
print(f"{'Store A':40s}  {'Store B':40s}  {'Similarity':>10s}")
print("=" * 95)
for prod_a, prod_b, score in matches:
    print(f"{prod_a['title'][:40]:<40s}  {prod_b['title'][:40]:<40s}  {score:>9.1%}")

In [ ]:
# Exact GTIN lookup — find a specific product by barcode
all_products = store_a + store_b
gtin_matches = shopextract.match_gtin("0850031234567", all_products)

print(f"Products matching GTIN 0850031234567: {len(gtin_matches)}\n")
for m in gtin_matches:
    print(f"  {m['title']:<45s}  ${m['price']:>7s}  GTIN: {m['gtin']}")

---

## 5. Marketplace Validation

Before listing products on a marketplace, you need to ensure they meet that platform's requirements. Each marketplace has different mandatory fields, character limits, and format rules.

`validate()` checks your product data against the rules for **Google Shopping**, **idealo**, **Amazon**, or **eBay** and returns a detailed report with errors and warnings.

In [ ]:
# Validate products for Google Shopping
sample_products = [
    {
        "title": "Classic Percale Sheet Set - White",
        "price": "149.00",
        "image_url": "https://example.com/sheets.jpg",
        "product_url": "https://example.com/products/sheets",
        "description": "Crisp, cool percale sheets in 100% long-staple cotton.",
        "gtin": "0850031234567",
    },
    {
        "title": "Luxe Sateen Pillow",
        "price": "49.00",
        # Missing image_url — should trigger an error
        "product_url": "https://example.com/products/pillow",
    },
    {
        "title": "Waffle Bath Towel",
        "price": "39.00",
        "image_url": "https://example.com/towel.jpg",
        "product_url": "https://example.com/products/towel",
        # Missing GTIN — should trigger a warning
    },
]

report = shopextract.validate(sample_products, marketplace="google_shopping")

print(f"Marketplace:  {report.marketplace}")
print(f"Total:        {report.total}")
print(f"Valid:        {report.valid}")
print(f"Invalid:      {report.invalid}")
print(f"Warnings:     {report.warnings}")
print(f"Pass Rate:    {report.pass_rate:.0f}%")

if report.issues:
    print(f"\nIssues Found:")
    print("-" * 70)
    for issue in report.issues:
        icon = "ERROR" if issue.severity == "error" else "WARN "
        print(f"  [{icon}] {issue.product_title}: {issue.field} — {issue.error}")

In [ ]:
# Validate the same products for idealo — stricter requirements
report_idealo = shopextract.validate(sample_products, marketplace="idealo")

print(f"idealo Validation: {report_idealo.valid}/{report_idealo.total} valid ({report_idealo.pass_rate:.0f}% pass rate)\n")
for issue in report_idealo.issues:
    icon = "ERROR" if issue.severity == "error" else "WARN "
    print(f"  [{icon}] {issue.product_title}: {issue.field} — {issue.error}")

In [ ]:
# Validate for Amazon — GTIN and brand are mandatory
report_amazon = shopextract.validate(sample_products, marketplace="amazon")

print(f"Amazon Validation: {report_amazon.valid}/{report_amazon.total} valid ({report_amazon.pass_rate:.0f}% pass rate)\n")
for issue in report_amazon.issues:
    icon = "ERROR" if issue.severity == "error" else "WARN "
    print(f"  [{icon}] {issue.product_title}: {issue.field} — {issue.error}")

---

## 6. Price Monitoring

`shopextract` can track product prices over time using lightweight SQLite snapshots. The workflow is:

1. **Snapshot** a store to capture current prices
2. Take another snapshot later (days, hours, minutes)
3. Run `changes()` to detect price increases, decreases, new products, and removed products
4. Run `price_history()` to see a specific product's price trajectory

Below we simulate this with two manual snapshots to demonstrate the API.

In [ ]:
import json
import sqlite3
import tempfile
from datetime import datetime, UTC, timedelta

# Create a temporary snapshot database with two snapshots
tmp_db = tempfile.NamedTemporaryFile(suffix=".db", delete=False)
tmp_db_path = tmp_db.name
tmp_db.close()

conn = sqlite3.connect(tmp_db_path)
conn.execute("""
    CREATE TABLE IF NOT EXISTS snapshots (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        domain TEXT NOT NULL,
        products_json TEXT NOT NULL,
        created_at TEXT NOT NULL
    )
""")

# Snapshot 1: Original prices (one week ago)
snapshot_v1 = [
    {"title": "Classic Percale Sheet Set",    "price": "149.00", "currency": "USD"},
    {"title": "Luxe Sateen Duvet Cover",      "price": "199.00", "currency": "USD"},
    {"title": "Down Alternative Comforter",    "price": "249.00", "currency": "USD"},
    {"title": "Waffle Bath Robe",              "price": "98.00",  "currency": "USD"},
]

# Snapshot 2: Current prices — some changed, one new, one removed
snapshot_v2 = [
    {"title": "Classic Percale Sheet Set",    "price": "129.00", "currency": "USD"},  # Price drop
    {"title": "Luxe Sateen Duvet Cover",      "price": "219.00", "currency": "USD"},  # Price increase
    {"title": "Down Alternative Comforter",    "price": "249.00", "currency": "USD"},  # Unchanged
    {"title": "Organic Cotton Towel Bundle",   "price": "89.00",  "currency": "USD"},  # New product
    # "Waffle Bath Robe" removed
]

ts_old = (datetime.now(UTC) - timedelta(days=7)).isoformat()
ts_new = datetime.now(UTC).isoformat()

conn.execute("INSERT INTO snapshots (domain, products_json, created_at) VALUES (?, ?, ?)",
             ("demo.store", json.dumps(snapshot_v1), ts_old))
conn.execute("INSERT INTO snapshots (domain, products_json, created_at) VALUES (?, ?, ?)",
             ("demo.store", json.dumps(snapshot_v2), ts_new))
conn.commit()
conn.close()

print(f"Created snapshot database at {tmp_db_path}")
print(f"  Snapshot 1: {len(snapshot_v1)} products ({ts_old[:10]})")
print(f"  Snapshot 2: {len(snapshot_v2)} products ({ts_new[:10]})")

In [ ]:
# Detect changes between the two snapshots
detected = shopextract.changes("demo.store", db_path=tmp_db_path)

print(f"Detected {len(detected)} changes\n")
for change in detected:
    if change.change_type.value == "price_change":
        direction = "dropped" if change.new_price < change.old_price else "increased"
        print(f"  PRICE {direction.upper():>9s}  {change.title}")
        print(f"                    ${change.old_price} -> ${change.new_price} {change.currency}\n")
    elif change.change_type.value == "new_product":
        print(f"  NEW PRODUCT       {change.title}")
        print(f"                    ${change.price} {change.currency}\n")
    elif change.change_type.value == "removed_product":
        print(f"  REMOVED           {change.title}")
        print(f"                    Last price: ${change.last_price} {change.currency}\n")

In [ ]:
# Price history for a specific product across all snapshots
history = shopextract.price_history("demo.store", "Classic Percale Sheet Set", db_path=tmp_db_path)

print("Price History: Classic Percale Sheet Set")
print("=" * 45)
for ts, price in history:
    print(f"  {ts.strftime('%Y-%m-%d %H:%M')}   ${price:.2f}")

# Clean up temp database
import os
os.unlink(tmp_db_path)

---

## 7. Export

Product data can be exported in multiple formats for downstream consumption:

| Format | Function | Use Case |
|---|---|---|
| **CSV** | `to_csv()` | Spreadsheets, data pipelines |
| **JSON** | `to_json()` | APIs, web applications |
| **Google Shopping XML** | `to_feed(format="google_shopping")` | Google Merchant Center feed |
| **Pandas DataFrame** | `to_dataframe()` | In-memory analysis |
| **Parquet** | `to_parquet()` | Columnar storage, big data |

All export functions accept a list of product dicts.

In [ ]:
import tempfile

# Sample products for export demos
export_products = [
    {
        "title": "Classic Percale Sheet Set - White",
        "price": "149.00",
        "currency": "USD",
        "image_url": "https://example.com/images/sheets-white.jpg",
        "product_url": "https://example.com/products/percale-sheets-white",
        "description": "Crisp, cool 100% long-staple cotton percale. 270 thread count.",
        "vendor": "HomeBasics",
        "sku": "PERC-SHT-WHT-Q",
        "gtin": "0850031234567",
        "in_stock": True,
        "condition": "new",
    },
    {
        "title": "Luxe Sateen Duvet Cover - Cream",
        "price": "199.00",
        "currency": "USD",
        "image_url": "https://example.com/images/duvet-cream.jpg",
        "product_url": "https://example.com/products/sateen-duvet-cream",
        "description": "Buttery smooth sateen weave with a subtle sheen. 480 thread count.",
        "vendor": "HomeBasics",
        "sku": "SATN-DVT-CRM-Q",
        "gtin": "0850031234890",
        "in_stock": True,
        "condition": "new",
    },
    {
        "title": "Down Alternative Comforter",
        "price": "249.00",
        "currency": "USD",
        "image_url": "https://example.com/images/comforter.jpg",
        "product_url": "https://example.com/products/down-alt-comforter",
        "description": "Hypoallergenic fill, all-season weight. Machine washable.",
        "vendor": "HomeBasics",
        "sku": "DOWN-ALT-CMF-Q",
        "in_stock": True,
        "condition": "new",
    },
]

In [ ]:
# Export to CSV
csv_path = tempfile.mktemp(suffix=".csv")
shopextract.to_csv(export_products, csv_path)

with open(csv_path) as f:
    lines = f.readlines()

print("CSV Export (first 5 lines):")
print("-" * 80)
for line in lines[:5]:
    print(line.rstrip()[:120])

In [ ]:
# Export to JSON
json_path = tempfile.mktemp(suffix=".json")
shopextract.to_json(export_products, json_path, indent=2)

with open(json_path) as f:
    content = f.read()

print("JSON Export (first product):")
print("-" * 50)
# Pretty-print just the first product
import json as _json
parsed = _json.loads(content)
print(_json.dumps(parsed[0], indent=2))

In [ ]:
# Export to Google Shopping XML feed
feed_path = tempfile.mktemp(suffix=".xml")
shopextract.to_feed(export_products, feed_path, format="google_shopping")

with open(feed_path) as f:
    lines = f.readlines()

print("Google Shopping XML Feed (first 20 lines):")
print("-" * 60)
for line in lines[:20]:
    print(line.rstrip())

In [ ]:
# Export to Pandas DataFrame for interactive analysis
df = shopextract.to_dataframe(export_products)

df[["title", "price", "currency", "vendor", "sku", "gtin"]]

---

## 8. Quality Scoring

`QualityScorer` evaluates how complete and useful each product record is on a 0.0 - 1.0 scale.

The scoring breakdown:
- **0.40** — Base score for having a title
- **+0.15** — Has a price
- **+0.15** — Has an image
- **+0.15** — Has a description
- **+0.15** — Has a SKU or external ID

A score of **1.0** means the product has all key fields populated. Products without a title score **0.0**.

In [ ]:
# Score individual products to see which fields contribute
scorer = shopextract.QualityScorer()

test_products = [
    {"title": "Complete Product", "price": "49.99", "image_url": "https://img.jpg", "description": "Great item", "sku": "ABC-123"},
    {"title": "Missing Image",   "price": "29.99", "description": "Nice product", "sku": "DEF-456"},
    {"title": "Bare Minimum",    "price": "9.99"},
    {"title": "Title Only"},
    {},  # Empty product — no title
]

print(f"{'Product':<25s}  {'Score':>5s}  Fields Present")
print("=" * 75)
for p in test_products:
    score = scorer.score_product(p)
    fields = []
    if p.get("title"):       fields.append("title")
    if p.get("price"):       fields.append("price")
    if p.get("image_url"):   fields.append("image")
    if p.get("description"): fields.append("desc")
    if p.get("sku"):         fields.append("sku")
    label = p.get("title", "(empty)")[:25]
    print(f"  {label:<25s}  {score:>4.2f}  {', '.join(fields) or '—'}")

In [ ]:
# Batch quality score — average across all products
batch_score = scorer.score_batch(test_products)
print(f"Batch Average Quality: {batch_score:.2f}")
print(f"Products Scored:       {len(test_products)}")

---

## 9. Duplicate Detection

`find_duplicates()` identifies duplicate products within a catalog using one of three methods:

| Method | How it works |
|---|---|
| `"title"` | Fuzzy string matching on product titles (configurable threshold) |
| `"gtin"` | Exact match on GTIN/EAN/UPC barcodes |
| `"sku"` | Exact match on SKU values |

Returns a list of `(index_a, index_b, similarity)` tuples for each duplicate pair found.

In [ ]:
# Create a catalog with intentional duplicates
catalog_with_dupes = [
    {"title": "Classic Percale Sheet Set - White",  "price": "149.00", "gtin": "0850031234567", "sku": "PERC-001"},
    {"title": "Luxe Sateen Duvet Cover - Cream",    "price": "199.00", "gtin": "0850031234890", "sku": "SATN-001"},
    {"title": "Classic Percale Sheet Set (White)",   "price": "149.00", "gtin": "0850031234567", "sku": "PERC-001-V2"},  # Title variant + same GTIN
    {"title": "Down Alternative Comforter",          "price": "249.00", "gtin": "0850031234111", "sku": "DOWN-001"},
    {"title": "Luxe Sateen Duvet - Cream",           "price": "209.00", "gtin": "0850031234999", "sku": "SATN-002"},     # Similar title, different GTIN
]

# Find duplicates by fuzzy title matching
title_dupes = shopextract.find_duplicates(catalog_with_dupes, method="title", threshold=0.8)

print(f"Title-Based Duplicates (threshold=0.8): {len(title_dupes)} pairs\n")
for idx_a, idx_b, sim in title_dupes:
    print(f"  [{idx_a}] {catalog_with_dupes[idx_a]['title']}")
    print(f"  [{idx_b}] {catalog_with_dupes[idx_b]['title']}")
    print(f"       Similarity: {sim:.1%}\n")

In [ ]:
# Find duplicates by exact GTIN match
gtin_dupes = shopextract.find_duplicates(catalog_with_dupes, method="gtin")

print(f"GTIN-Based Duplicates: {len(gtin_dupes)} pairs\n")
for idx_a, idx_b, sim in gtin_dupes:
    gtin = catalog_with_dupes[idx_a]["gtin"]
    print(f"  [{idx_a}] {catalog_with_dupes[idx_a]['title']}")
    print(f"  [{idx_b}] {catalog_with_dupes[idx_b]['title']}")
    print(f"       GTIN: {gtin}  (exact match)\n")

---

## Summary

This notebook demonstrated the core capabilities of `shopextract`:

1. **Platform Detection** — Automatic identification of Shopify, WooCommerce, Magento, BigCommerce, Shopware, and custom stores
2. **Product Extraction** — Tiered strategy (API > crawl > CSS) that adapts to each store's capabilities
3. **Catalog Analysis** — Price statistics, brand distributions, and completeness scoring
4. **Product Matching** — Fuzzy title matching and exact GTIN lookup across catalogs
5. **Marketplace Validation** — Pre-listing checks for Google Shopping, idealo, Amazon, and eBay
6. **Price Monitoring** — Snapshot-based change detection and price history tracking
7. **Multi-Format Export** — CSV, JSON, Google Shopping XML, Pandas DataFrames, and Parquet
8. **Quality Scoring** — Per-product and batch-level data quality assessment
9. **Duplicate Detection** — Title similarity, GTIN, and SKU-based deduplication

### Next Steps

- **`shopextract.watch(url)`** — Real-time async generator that yields changes as they happen
- **`shopextract.compare(query, stores)`** — Cross-store price comparison for a specific product
- **`shopextract.from_feed(feed_url)`** — Import from Google Shopping XML/CSV feeds directly
- **`shopextract.to_parquet(products, path)`** — Columnar export for large-scale analytics

For full API documentation, see the [source code](../src/shopextract/__init__.py).